# LeetCode 3181 — Maximum Total Reward Using Operations II bằng Python

- Link: https://leetcode.com/problems/maximum-total-reward-using-operations-ii/
- Độ khó: **Hard**
- Chủ đề: **DP tập tổng đạt được (subset-sum) dạng bitset** — sắp xếp tăng dần, giá trị `v` chỉ được cộng vào các tổng `x < v`: `f |= (f & ((1 << v) − 1)) << v`; số nguyên lớn của Python đóng vai bitset

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxTotalReward(self, rewardValues: List[int]) -> int:
```

## 1. Mục tiêu học

1. Nhận ra: nếu chọn một tập phần thưởng, có thể lấy hết **khi và chỉ khi** lấy theo thứ tự **tăng dần** thoả mãn — nên chỉ cần xét các tập và thứ tự tăng dần.
2. Mỗi giá trị dùng tối đa một lần và hai giá trị bằng nhau không thể cùng dùng ⇒ **khử trùng lặp**.
3. DP tập tổng: `reach` = tập các tổng `x` đạt được; xử lý `v` tăng dần: `reach ∪= {x + v : x ∈ reach, x < v}`.
4. Cài bằng **bitset** (số nguyên Python): `f |= (f & mask_v) << v` — mỗi bước `O(M / 64)` máy.
5. Tổng luôn `< 2 · max` ⇒ bitset dài `≤ 10^5` bit.

## 2. Đề bài & ví dụ

Cho mảng `rewardValues`. Ban đầu tổng `x = 0`, mọi chỉ số chưa đánh dấu. Mỗi thao tác chọn chỉ số `i` chưa đánh dấu; nếu `rewardValues[i] > x` thì `x += rewardValues[i]` và đánh dấu `i`. Tìm `x` lớn nhất.

**Ví dụ 1:** `[1, 1, 3, 3]` → `4` (lấy `1` rồi `3`).

**Ví dụ 2:** `[1, 6, 4, 3, 2]` → `11` (lấy `1, 4, 6`).

**Ràng buộc:** `1 ≤ n ≤ 5·10⁴`; `1 ≤ rewardValues[i] ≤ 5·10⁴`.

In [1]:
from typing import List
import itertools, random, time

examples = [([1, 1, 3, 3], 4), ([1, 6, 4, 3, 2], 11)]
for a, exp in examples:
    print(a, '-> kỳ vọng', exp)

[1, 1, 3, 3] -> kỳ vọng 4
[1, 6, 4, 3, 2] -> kỳ vọng 11


## 3. Kiến thức nền

### 3.1. Chỉ cần xét thứ tự tăng dần

Giả sử lấy được dãy `v₁, v₂, …, v_k` (theo thứ tự lấy). Điều kiện: `v_j > v₁ + … + v_{j−1}` ⇒ `v_j > v_{j−1}` ⇒ dãy **tăng ngặt**. Vậy một tập lấy được thì phải lấy theo thứ tự tăng dần, và mọi giá trị trong tập khác nhau.

### 3.2. Tổng luôn nhỏ hơn `2 · max`

Phần tử cuối `v_k ≤ M = max` và trước đó `x < v_k` ⇒ tổng cuối `x + v_k < 2 v_k ≤ 2M`. Bitset chỉ cần `2M` bit.

In [2]:
def show_bits(f, upto):
    return [x for x in range(upto) if f >> x & 1]

f = 1                                  # chỉ có tổng 0
for v in [1, 2, 3, 4, 6]:
    f |= (f & ((1 << v) - 1)) << v
    print(f'sau v = {v}: tổng đạt được = {show_bits(f, 12)}')

sau v = 1: tổng đạt được = [0, 1]
sau v = 2: tổng đạt được = [0, 1, 2, 3]
sau v = 3: tổng đạt được = [0, 1, 2, 3, 4, 5]
sau v = 4: tổng đạt được = [0, 1, 2, 3, 4, 5, 6, 7]
sau v = 6: tổng đạt được = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Đây là subset-sum có **ràng buộc thứ tự**: một giá trị `v` chỉ "nối" được vào các tổng **nhỏ hơn** nó. Xử lý giá trị theo thứ tự tăng dần, ta chắc chắn mọi tổng `x` đã có đều dùng các giá trị `< v`, nên nối `v` vào sau là hợp lệ.

Ví dụ 2: sau `1, 2, 3, 4` các tổng là `{0, 1, 2, 3, 4, 5, 6, 7}`; với `v = 6` chỉ các tổng `< 6` được nối ⇒ thêm `{6, …, 11}` ⇒ max `11`.

## 5. Cách 1 — Brute Force: thử mọi tập con, lấy theo thứ tự tăng dần

In [3]:
def brute(vals):
    best = 0; n = len(vals)
    for mask in range(1 << n):
        x = 0; ok = True
        for v in sorted(vals[i] for i in range(n) if mask >> i & 1):
            if v > x:
                x += v
            else:
                ok = False; break
        if ok:
            best = max(best, x)
    return best

for a, exp in examples:
    print('brute:', brute(a), '| kỳ vọng', exp)

brute: 4 | kỳ vọng 4
brute: 11 | kỳ vọng 11


**Độ phức tạp:** `O(2^n · n log n)` — chỉ cho `n ≤ 14`.

## 6. Ý tưởng thuật toán / bước đột phá

```
vals = sorted(set(rewardValues))
f = 1                                   # bit x bật ⇔ tổng x đạt được
for v in vals:
    f |= (f & ((1 << v) - 1)) << v      # chỉ các tổng < v được cộng v
return f.bit_length() - 1
```

Mỗi bước là vài phép toán trên số nguyên `≤ 10^5` bit ⇒ tổng khoảng `n · 10^5 / 64 ≈ 8·10⁷` phép máy — vài giây trong C của Python. Cắt tỉa thêm ở cách 4.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP mảng boolean `O(n · M)` (bản I)

In [4]:
def dp_bool(vals):
    vals = sorted(set(vals)); M = vals[-1]
    can = [False] * (2 * M); can[0] = True
    for v in vals:
        for x in range(v - 1, -1, -1):          # chỉ x < v; duyệt giảm không cần thiết vì x + v >= v > x
            if can[x]:
                can[x + v] = True
    return max(x for x in range(2 * M) if can[x])

for a, exp in examples:
    print('dp bool:', dp_bool(a), '| kỳ vọng', exp)

dp bool: 4 | kỳ vọng 4
dp bool: 11 | kỳ vọng 11


Vì `x + v ≥ v > x`, các ô được ghi luôn nằm **ngoài** vùng đang đọc `[0, v)` ⇒ thứ tự duyệt không quan trọng. `O(n · M)` = `2.5·10⁹` xấu nhất với bản II — chỉ dùng cho dữ liệu vừa.

### 7.2. Cách 3 — Bitset bằng số nguyên lớn (lời giải chính)

In [5]:
def bitset(vals):
    f = 1
    for v in sorted(set(vals)):
        f |= (f & ((1 << v) - 1)) << v
    return f.bit_length() - 1

for a, exp in examples:
    print('bitset:', bitset(a), '| kỳ vọng', exp)

bitset: 4 | kỳ vọng 4
bitset: 11 | kỳ vọng 11


### 7.3. Cách 4 — Bitset + cắt tỉa "đạt trần `2M − 1`"

Đáp án `≤ 2M − 1` (mục 3.2). Đạt trần khi trước `M` có tổng `M − 1`. Hai cách nhận biết nhanh:

- `M − 1` có trong mảng ⇒ lấy `M − 1` rồi `M` ⇒ `2M − 1`.
- Có hai giá trị `a < b` khác `M` với `a + b = M − 1` ⇒ lấy `a, b, M` ⇒ `2M − 1`.

Ngoài ra, trong vòng bitset, nếu đã thấy bit `M − 1` bật trước khi xử lý `M` thì dừng sớm.

In [6]:
def bitset_pruned(vals):
    s = sorted(set(vals)); M = s[-1]; S = set(s)
    if M - 1 in S:
        return 2 * M - 1
    for a in s:
        b = M - 1 - a
        if b > a and b in S and b != M:
            return 2 * M - 1
    f = 1
    for v in s:
        f |= (f & ((1 << v) - 1)) << v
        if f >> (M - 1) & 1:
            return 2 * M - 1
    return f.bit_length() - 1

for a, exp in examples:
    print('bitset + cắt tỉa:', bitset_pruned(a), '| kỳ vọng', exp)

bitset + cắt tỉa: 4 | kỳ vọng 4
bitset + cắt tỉa: 11 | kỳ vọng 11


### 7.4. Cách 5 — Chỉ cần tổng tốt nhất `< M` trước khi lấy `M`

Đáp án `= M + max{x đạt được bằng các giá trị < M : x < M}` — vì luôn có lợi khi lấy `M` sau cùng (nếu `x < M`, thêm `M` tăng tổng). Ta chạy bitset trên các giá trị `< M`, rồi lấy bit cao nhất trong `[0, M)`.

In [7]:
def last_is_max(vals):
    s = sorted(set(vals)); M = s[-1]
    f = 1
    for v in s[:-1]:
        f |= (f & ((1 << v) - 1)) << v
    return M + (f & ((1 << M) - 1)).bit_length() - 1

for a, exp in examples:
    print('lấy M cuối cùng:', last_is_max(a), '| kỳ vọng', exp)

lấy M cuối cùng: 4 | kỳ vọng 4
lấy M cuối cùng: 11 | kỳ vọng 11


## 8. TRACE từng bước

Trace cách 3 trên `[2, 3, 4, 9]`: in tập tổng đạt được sau mỗi giá trị.

In [8]:
def trace(vals):
    f = 1
    for v in sorted(set(vals)):
        low = f & ((1 << v) - 1)
        f |= low << v
        print(f'v = {v}: tổng < {v} có thể nối: {show_bits(low, v)} -> tập tổng: {show_bits(f, 2 * max(vals))}')
    print('đáp án:', f.bit_length() - 1, '| brute:', brute(vals))

trace([2, 3, 4, 9])

v = 2: tổng < 2 có thể nối: [0] -> tập tổng: [0, 2]
v = 3: tổng < 3 có thể nối: [0, 2] -> tập tổng: [0, 2, 3, 5]
v = 4: tổng < 4 có thể nối: [0, 2, 3] -> tập tổng: [0, 2, 3, 4, 5, 6, 7]
v = 9: tổng < 9 có thể nối: [0, 2, 3, 4, 5, 6, 7] -> tập tổng: [0, 2, 3, 4, 5, 6, 7, 9, 11, 12, 13, 14, 15, 16]
đáp án: 16 | brute: 16


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def maxTotalReward(self, rewardValues: List[int]) -> int:
        vals = sorted(set(rewardValues))
        M = vals[-1]
        present = set(vals)
        if M - 1 in present:
            return 2 * M - 1
        f = 1
        for v in vals:
            f |= (f & ((1 << v) - 1)) << v
            if (f >> (M - 1)) & 1:
                return 2 * M - 1
        return f.bit_length() - 1

for a, exp in examples:
    got = Solution().maxTotalReward(a)
    print(got, 'OK' if got == exp else 'SAI')

4 OK
11 OK


## 10. Vì sao thuật toán đúng

1. **Tập lấy được ⇔ dãy tăng dần thoả `v_j > Σ_{i<j} v_i`** (mục 3.1) ⇒ có thể bỏ trùng lặp và xử lý theo thứ tự tăng.
2. **Bất biến:** sau khi xử lý các giá trị `≤ v`, bit `x` bật ⇔ có một dãy hợp lệ gồm các giá trị `≤ v` có tổng `x`. Bước `v`: một dãy hợp lệ kết thúc bằng `v` có dạng (dãy các giá trị `< v` tổng `x`) + `v` với điều kiện `v > x` ⇔ bit `x` bật trong `f & ((1 << v) − 1)`.
3. **Cắt tỉa:** đáp án `< 2M` nên `2M − 1` là trần; nếu đã đạt tổng `M − 1` (chỉ dùng giá trị `< M`, vì mọi tổng chứa `M` đều `≥ M`), lấy thêm `M` cho `2M − 1`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Brute | `O(2^n n log n)` | `O(n)` |
| 2. DP boolean | `O(n · M)` | `O(M)` |
| 3. Bitset | `O(n · M / w)` | `O(M / w)` |
| 4. Bitset + cắt tỉa | như 3, thường sớm hơn | `O(M / w)` |
| 5. Lấy `M` sau cùng | như 3 | `O(M / w)` |

(`w` = độ rộng từ máy, 64.)

### 11.1. Đo thực nghiệm chi phí một bước bitset

Một bước `f |= (f & mask) << v` gồm ba phép trên số nguyên dài `≈ 2M` bit: AND, dịch trái, OR. Mỗi phép tuyến tính theo số "chữ số" (30 bit/chữ số trong CPython) ⇒ khoảng `2M / 30 ≈ 3.300` chữ số khi `M = 5·10⁴`. Với `n = 5·10⁴` giá trị khác nhau, tổng cỡ `1.6·10⁸` thao tác chữ số trong C — nhanh hơn nhiều so với vòng lặp Python `O(n · M)`.

In [10]:
M = 5 * 10 ** 4
f = (1 << (2 * M)) - 1                      # bitset "đầy" để đo trường hợp xấu
t = time.time()
for v in range(1, 2001):
    g = f | ((f & ((1 << v) - 1)) << v)
per_step = (time.time() - t) / 2000
print(f'thời gian trung bình một bước bitset (2M = {2 * M} bit): {per_step * 1e6:.1f} µs')
print(f'ước lượng cho 5·10⁴ bước: {per_step * 5e4:.2f} s')

thời gian trung bình một bước bitset (2M = 100000 bit): 0.9 µs
ước lượng cho 5·10⁴ bước: 0.05 s


## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [11]:
impls = {'dp_bool': dp_bool, 'bitset': bitset, 'pruned': bitset_pruned, 'last_max': last_is_max,
         'submit': lambda a: Solution().maxTotalReward(a)}
edge = [([5], 5), ([7, 7, 7], 7), ([1, 2], 3), ([2, 3], 5), ([1, 2, 3], 5), ([3, 5, 9], 17), ([1, 3, 6, 12], 22)]
for a, exp in examples + edge:
    assert brute(a) == exp, (a, brute(a))
    for name, f in impls.items():
        assert f(a) == exp, (name, a)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


Giải thích: `[1, 2, 3]` — `1 + 2 = 3` không lớn hơn ngặt `3` ⇒ không lấy được cả ba; tốt nhất `2 + 3 = 5`. `[3, 5, 9]`: `3 + 5 = 8 < 9` ⇒ lấy cả ba, `17`. `[1, 3, 6, 12]`: `1 + 3 + 6 = 10 < 12` ⇒ `22`.

### 12.2. Đối chiếu ngẫu nhiên với brute

In [12]:
rng = random.Random(3181)
cnt = 0
for _ in range(400):
    n = rng.randint(1, 11); V = rng.choice([5, 12, 40])
    a = [rng.randint(1, V) for _ in range(n)]
    exp = brute(a)
    for name, f in impls.items():
        assert f(a) == exp, (name, a)
    cnt += 1
print('Khớp brute trên', cnt, 'mảng ngẫu nhiên.')

Khớp brute trên 400 mảng ngẫu nhiên.


In [13]:
# n vừa: so các cách với DP boolean
for _ in range(30):
    a = [rng.randint(1, 3000) for _ in range(rng.randint(1, 200))]
    x = dp_bool(a)
    assert bitset(a) == bitset_pruned(a) == last_is_max(a) == Solution().maxTotalReward(a) == x
print('Các cách bitset khớp DP boolean trên mảng tới 200 phần tử, giá trị ≤ 3000.')

Các cách bitset khớp DP boolean trên mảng tới 200 phần tử, giá trị ≤ 3000.


### 12.3. Ca lớn `n = 5·10⁴`, giá trị tới `5·10⁴`

In [14]:
N = 5 * 10 ** 4
big = {
    'ngẫu nhiên': [rng.randint(1, N) for _ in range(N)],
    'chẵn (không cắt tỉa sớm)': [2 * i for i in range(1, N // 2 + 1)],
    'toàn khác nhau 1..N': list(range(1, N + 1)),
}
for name, a in big.items():
    t = time.time(); x = Solution().maxTotalReward(a); t1 = time.time() - t
    t = time.time(); y = bitset(a); t2 = time.time() - t
    assert x == y
    print(f'{name:26s}: {x:6d} | nộp {t1:.2f}s | bitset thuần {t2:.2f}s')
assert Solution().maxTotalReward([2 * i for i in range(1, N // 2 + 1)]) == 2 * N - 2
print('Toàn số chẵn: tổng lẻ không đạt được, trần 2M − 2 = 2N − 2: đúng.')

ngẫu nhiên                :  99999 | nộp 0.01s | bitset thuần 0.09s


chẵn (không cắt tỉa sớm)  :  99998 | nộp 0.08s | bitset thuần 0.08s


toàn khác nhau 1..N       :  99999 | nộp 0.00s | bitset thuần 0.13s


Toàn số chẵn: tổng lẻ không đạt được, trần 2M − 2 = 2N − 2: đúng.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam: duyệt tăng dần, gặp được thì lấy

In [15]:
def wrong_greedy(vals):
    x = 0
    for v in sorted(vals):
        if v > x:
            x += v                                                   # SAI
    return x

print('sai :', wrong_greedy(examples[1][0]), '| đúng:', Solution().maxTotalReward(examples[1][0]))

sai : 7 | đúng: 11


**Vì sao sai:** lấy `2` sau `1` (tổng 3) chặn mất `3`; tối ưu là bỏ `2, 3` để lấy `1, 4, 6`.

### Lỗi 2 — Cho phép `v ≥ x` (mặt nạ `(1 << (v + 1)) − 1`)

In [16]:
def wrong_mask(vals):
    f = 1
    for v in sorted(set(vals)):
        f |= (f & ((1 << (v + 1)) - 1)) << v                         # SAI: gồm cả x = v
    return f.bit_length() - 1

print('sai :', wrong_mask([1, 2, 3]), '| đúng:', Solution().maxTotalReward([1, 2, 3]))

sai : 6 | đúng: 5


**Vì sao sai:** đề yêu cầu **lớn hơn ngặt**. Với `[1, 2, 3]`, tổng `3 = 1 + 2` không cho phép lấy `3`.

### Lỗi 3 — Không sắp xếp trước khi chạy bitset

In [17]:
def wrong_unsorted(vals):
    f = 1
    for v in dict.fromkeys(vals):                                    # SAI: giữ thứ tự đầu vào
        f |= (f & ((1 << v) - 1)) << v
    return f.bit_length() - 1

print('sai :', wrong_unsorted(examples[1][0]), '| đúng:', Solution().maxTotalReward(examples[1][0]))

sai : 7 | đúng: 11


**Vì sao sai:** xử lý `6` trước `4, 3, 2` thì các tổng chứa `6` không còn được nối thêm giá trị nhỏ — nhưng ở bước sau, `4` lại được nối vào các tổng nhỏ "sau" `6`, không tương ứng thứ tự lấy hợp lệ. Bất biến chỉ đúng khi xử lý tăng dần.

### Lỗi 4 — Mảng DP chỉ dài `M + 1` (quên tổng có thể tới `2M − 1`)

In [18]:
def wrong_short_array(vals):
    vals = sorted(set(vals)); M = vals[-1]
    can = [False] * (M + 1); can[0] = True                          # SAI: phải là 2M
    for v in vals:
        for x in range(v):
            if can[x] and x + v <= M:
                can[x + v] = True
    return max(x for x in range(M + 1) if can[x])

print('sai :', wrong_short_array(examples[1][0]), '| đúng:', Solution().maxTotalReward(examples[1][0]))

sai : 6 | đúng: 11


**Vì sao sai:** tổng cuối thường **vượt** `M` (ví dụ `5 + 6 = 11`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chỉ được lấy khi lớn hơn tổng hiện tại" ⇒ dãy tăng, xử lý tăng dần, subset-sum có điều kiện.
- Python: số nguyên lớn là bitset miễn phí — `f |= (f & mask) << v`.
- Tìm cận trên đơn giản (`2M − 1`) để cắt tỉa.
- Bản I (`n, v ≤ 2000`) cho phép DP boolean; bản II buộc bitset.

## 15. Biến thể của bài

1. **LC 3180 (bản I):** cùng lời giải, ràng buộc nhỏ.
2. Điều kiện `≥` thay vì `>` ⇒ đổi mặt nạ thành `(1 << (v + 1)) − 1` và **không** được khử trùng lặp: lấy `v` rồi lấy tiếp một bản sao `v` nữa là hợp lệ (`v ≥ v`), nên mỗi bản sao xử lý riêng (vẫn theo thứ tự tăng dần).
3. Đếm số tổng khác nhau đạt được ⇒ `popcount(f)`.
4. Tối đa **số lượng** phần thưởng lấy được (thay vì tổng) ⇒ DP khác (tham lam lấy nhỏ nhất có thể là đúng — kiểm chứng ở dưới).

In [19]:
# Biến thể 2: điều kiện v >= x (không khử trùng hoàn toàn: mỗi phần tử dùng một lần)
def ge_version(vals):
    f = 1
    for v in sorted(vals):                       # giữ trùng lặp
        f |= (f & ((1 << (v + 1)) - 1)) << v
    return f.bit_length() - 1

def ge_brute(vals):
    best = 0; n = len(vals)
    for mask in range(1 << n):
        x = 0; ok = True
        for v in sorted(vals[i] for i in range(n) if mask >> i & 1):
            if v >= x: x += v
            else: ok = False; break
        if ok: best = max(best, x)
    return best

# Biến thể 4: số lượng phần thưởng nhiều nhất — tham lam lấy nhỏ nhất có thể
def max_count(vals):
    x = cnt = 0
    for v in sorted(vals):
        if v > x:
            x += v; cnt += 1
    return cnt

def max_count_brute(vals):
    best = 0; n = len(vals)
    for mask in range(1 << n):
        x = 0; ok = True; c = 0
        for v in sorted(vals[i] for i in range(n) if mask >> i & 1):
            if v > x: x += v; c += 1
            else: ok = False; break
        if ok: best = max(best, c)
    return best

for _ in range(200):
    a = [rng.randint(1, 12) for _ in range(rng.randint(1, 9))]
    assert ge_version(a) == ge_brute(a)
    assert max_count(a) == max_count_brute(a)
print('Biến thể "≥" (giữ trùng) và "nhiều phần thưởng nhất" (tham lam) khớp brute.')

Biến thể "≥" (giữ trùng) và "nhiều phần thưởng nhất" (tham lam) khớp brute.


(Biến thể 4: tham lam đúng vì nếu một dãy tối ưu có phần tử đầu tiên khác lựa chọn tham lam, thay bằng phần tử nhỏ hơn không làm hỏng các điều kiện sau — tổng nhỏ đi.)

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Thử tập con | mũ | Oracle |
| 2 | DP boolean | `O(nM)` | Bản I |
| 3 | Bitset | `O(nM/w)` | **Lõi lời giải** |
| 4 | Bitset + cắt tỉa | `O(nM/w)` | **Nộp bài** |
| 5 | Lấy `M` cuối | `O(nM/w)` | Góc nhìn khác |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `[5]`.

**Bài 2 (tính tay):** `[2, 3, 4, 9]`.

**Bài 3 (điền TODO):** trả về **dãy giá trị** (theo thứ tự lấy) đạt tổng lớn nhất.

**Bài 4 (tự viết lại):** đếm số **tổng khác nhau** (kể cả 0) có thể đạt được.

**Bài 5 (mini challenge):** cho thêm số nguyên `T`; tìm tổng lớn nhất **không vượt quá** `T`.

In [20]:
def best_sequence_todo(vals):
    # TODO
    raise NotImplementedError

def count_totals_todo(vals):
    # TODO
    raise NotImplementedError

def best_at_most_todo(vals, T):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Chỉ một phần tử. (2) `5 > 0`. (3) `5`.

**Bài 2.** (1) Trước `9` các tổng `< 9` đạt được: `0, 2, 3, 4, 5, 6, 7`. (2) Lớn nhất `7` (`3 + 4`). (3) `16`.

**Bài 3.** (1) Lưu với mỗi tổng `x` giá trị cuối cùng `last[x]` tạo ra nó (lần đầu bật). (2) Truy ngược `x → x − last[x]`. (3) Đảo dãy; kiểm từng bước `v > x`.

**Bài 4.** (1) Đáp án là số bit bật của `f`. (2) `bin(f).count('1')` hoặc `f.bit_count()`. (3) Kiểm bằng brute liệt kê tập tổng.

**Bài 5.** (1) Bitset như cũ. (2) Cắt `f & ((1 << (T + 1)) − 1)`. (3) Bit cao nhất.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2, 3
assert Solution().maxTotalReward([5]) == 5
assert Solution().maxTotalReward([2, 3, 4, 9]) == 16

def best_sequence(vals):
    s = sorted(set(vals)); M = s[-1]
    last = [0] * (2 * M); reach = [False] * (2 * M); reach[0] = True
    for v in s:
        for x in range(v):
            if reach[x] and not reach[x + v]:
                reach[x + v] = True; last[x + v] = v
    x = max(i for i in range(2 * M) if reach[i]); seq = []
    while x:
        seq.append(last[x]); x -= last[x]
    return seq[::-1]

for _ in range(300):
    a = [rng.randint(1, 20) for _ in range(rng.randint(1, 9))]
    seq = best_sequence(a); x = 0
    for v in seq:
        assert v > x; x += v
    from collections import Counter
    assert not (Counter(seq) - Counter(a)) and x == Solution().maxTotalReward(a)
print('Bài 1: 5 | Bài 2: 16 | Bài 3: dãy hợp lệ đạt tối ưu, ví dụ 2 ->', best_sequence(examples[1][0]))

Bài 1: 5 | Bài 2: 16 | Bài 3: dãy hợp lệ đạt tối ưu, ví dụ 2 -> [2, 3, 6]


In [22]:
# Bài 4, 5
def count_totals(vals):
    f = 1
    for v in sorted(set(vals)):
        f |= (f & ((1 << v) - 1)) << v
    return bin(f).count('1')

def best_at_most(vals, T):
    f = 1
    for v in sorted(set(vals)):
        f |= (f & ((1 << v) - 1)) << v
    return (f & ((1 << (T + 1)) - 1)).bit_length() - 1

def all_totals(vals):
    res = set(); n = len(vals)
    for mask in range(1 << n):
        x = 0; ok = True
        for v in sorted(vals[i] for i in range(n) if mask >> i & 1):
            if v > x: x += v
            else: ok = False; break
        if ok: res.add(x)
    return res

for _ in range(200):
    a = [rng.randint(1, 15) for _ in range(rng.randint(1, 9))]; T = rng.randint(0, 30)
    tot = all_totals(a)
    assert count_totals(a) == len(tot)
    assert best_at_most(a, T) == max(x for x in tot if x <= T)
print('Bài 4 (số tổng khác nhau) và Bài 5 (tổng lớn nhất ≤ T) khớp brute.')

Bài 4 (số tổng khác nhau) và Bài 5 (tổng lớn nhất ≤ T) khớp brute.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
vals = sorted(set(a)); M = vals[-1]
if M - 1 in set(vals): return 2 * M - 1
f = 1
for v in vals:
    f |= (f & ((1 << v) - 1)) << v
    if f >> (M - 1) & 1: return 2 * M - 1
return f.bit_length() - 1
```

**Checklist:**

- [ ] Sắp xếp **tăng dần** trước khi chạy bitset?
- [ ] Mặt nạ `(1 << v) − 1` (chỉ `x < v`, ngặt)?
- [ ] Khử trùng lặp (không bắt buộc nhưng nhanh hơn)?
- [ ] Mảng/bitset đủ dài `2M`?